# Basics: thermo, velocity triangles, cascade parameters

A walk-through of the basic functions of `meanline`.

**Model level:** mean-line (1D) at a single mean radius. Nothing here is a 3D
result, and nothing is yet validated against published data. The checks are
hand calculations and ideal-gas cross-checks.

**Sign convention:** flow angles are measured from the axial direction.
`c_theta` is positive in the direction of blade motion and `w_theta` is
positive against it, so the triangle closes as `U = c_theta + w_theta`.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from CoolProp.CoolProp import PropsSI

from meanline.cascade import de_haller_number, diffusion_factor
from meanline.thermo import ThermoState, static_from_total, total_from_static
from meanline.velocity_triangles import (
    degree_of_reaction,
    euler_work,
    flow_coefficient,
    stage_loading_coefficient,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Thermodynamic states: static to total

A stagnation process is isentropic. So the total state has enthalpy
`h0 = h + c**2 / 2` and the same entropy as the static state. CoolProp then
returns `T0`, `p0` and `rho0` from `(h0, s)`. No `gamma` or `R` is needed,
which keeps the method valid for real gases.

In [ ]:
FLUID = "Air"
static = ThermoState.state_from_pT(FLUID, p=101_325.0, T=300.0)
c = 100.0  # m/s

total = total_from_static(static, c)
cp = PropsSI("CPMASS", "P", static.p, "T", static.T, FLUID)

print(f"static: T = {static.T:.3f} K, p = {static.p:.0f} Pa, a = {static.a:.1f} m/s")
print(f"total : T0 = {total.T:.3f} K, p0 = {total.p:.0f} Pa")
print(f"T0 - T = {total.T - static.T:.3f} K")
print(f"low-speed estimate c^2 / (2 cp) = {c**2 / (2 * cp):.3f} K")

back = static_from_total(total, c)
print(f"round trip: dT = {back.T - static.T:.2e} K, dp = {back.p - static.p:.2e} Pa")

### Real gas versus ideal gas

Sweep the Mach number at fixed static conditions. The ideal-gas isentropic
relations use `gamma = cp / cv` taken from CoolProp at the static state. They
are a cross-check only and are not used by the model.

In [ ]:
cv = PropsSI("CVMASS", "P", static.p, "T", static.T, FLUID)
gamma = cp / cv

mach = np.linspace(0.0, 0.8, 41)
t_ratio = np.empty_like(mach)
p_ratio = np.empty_like(mach)
for i, m in enumerate(mach):
    tot = total_from_static(static, m * static.a)
    t_ratio[i] = tot.T / static.T
    p_ratio[i] = tot.p / static.p

t_ideal = 1.0 + 0.5 * (gamma - 1.0) * mach**2
p_ideal = t_ideal ** (gamma / (gamma - 1.0))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(mach, t_ratio, label="real gas (CoolProp)")
axes[0].plot(mach, t_ideal, "--", label="ideal gas, constant gamma")
axes[0].set_xlabel("Mach number [-]")
axes[0].set_ylabel("T0 / T [-]")
axes[0].legend()

axes[1].plot(mach, p_ratio, label="real gas (CoolProp)")
axes[1].plot(mach, p_ideal, "--", label="ideal gas, constant gamma")
axes[1].set_xlabel("Mach number [-]")
axes[1].set_ylabel("p0 / p [-]")
axes[1].legend()

fig.suptitle("Air, static 300 K and 1 atm: total-to-static ratios")
fig.tight_layout()
plt.show()

print(f"max relative difference in T0/T: {np.max(np.abs(t_ratio / t_ideal - 1)):.2e}")
print(f"max relative difference in p0/p: {np.max(np.abs(p_ratio / p_ideal - 1)):.2e}")

At 1 atm and 300 K, air is close to an ideal gas, so the two curves almost
overlap. The real-gas method matters at higher pressures and for other fluids,
where constant-`gamma` formulas drift.

## 2. Velocity triangles and stage parameters

A constant-radius rotor with a symmetric (50 % reaction) triangle:
`alpha1 = beta2` and `alpha2 = beta1`.

In [ ]:
u = 300.0  # blade speed [m/s]
c_x = 150.0  # axial velocity, constant across the rotor [m/s]
c_theta_1, c_theta_2 = 75.0, 225.0  # absolute swirl [m/s]

w_theta_1, w_theta_2 = u - c_theta_1, u - c_theta_2  # triangle closure
w_1 = math.hypot(c_x, w_theta_1)
w_2 = math.hypot(c_x, w_theta_2)

alpha_1, alpha_2 = math.atan2(c_theta_1, c_x), math.atan2(c_theta_2, c_x)
beta_1, beta_2 = math.atan2(w_theta_1, c_x), math.atan2(w_theta_2, c_x)

work = euler_work(u, u, c_theta_1, c_theta_2)
phi = flow_coefficient(c_x, u)
psi = stage_loading_coefficient(work, u)
reaction = degree_of_reaction(w_1, w_2, work)

print(f"Euler work       = {work / 1e3:.2f} kJ/kg")
print(f"flow coefficient = {phi:.3f}")
print(f"loading coeff.   = {psi:.3f}")
print(f"reaction         = {reaction:.3f}")
angles = {"alpha1": alpha_1, "alpha2": alpha_2, "beta1": beta_1, "beta2": beta_2}
for name, angle in angles.items():
    print(f"{name} = {math.degrees(angle):.2f} deg")

In [ ]:
def draw_triangle(ax, c_x, c_theta, u, title):
    """Draw c, w and U from a common origin; axial flow points down."""
    origin = (0.0, 0.0)
    c_tip = (c_theta, -c_x)
    w_tip = (c_theta - u, -c_x)
    arrow = {"arrowstyle": "->", "lw": 1.8}
    ax.annotate("", xy=c_tip, xytext=origin, arrowprops={**arrow, "color": "C0"})
    ax.annotate("", xy=w_tip, xytext=origin, arrowprops={**arrow, "color": "C3"})
    ax.annotate("", xy=c_tip, xytext=w_tip, arrowprops={**arrow, "color": "C2"})
    ax.text(c_tip[0] / 2 + 8, -c_x / 2, "c", color="C0")
    ax.text(w_tip[0] / 2 - 18, -c_x / 2, "w", color="C3")
    ax.text((c_tip[0] + w_tip[0]) / 2, -c_x - 22, "U", color="C2", ha="center")
    ax.set_xlim(-u - 20, u)
    ax.set_ylim(-c_x - 50, 30)
    ax.set_aspect("equal")
    ax.set_xlabel("tangential [m/s], blade motion to the right")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
draw_triangle(axes[0], c_x, c_theta_1, u, "Rotor inlet (station 1)")
draw_triangle(axes[1], c_x, c_theta_2, u, "Rotor exit (station 2)")
axes[0].set_ylabel("axial [m/s]")
fig.tight_layout()
plt.show()

The two triangles are mirror images, which is what a 50 % reaction stage looks
like: `w1 = c2`, `w2 = c1`, and the reaction comes out as 0.5.

## 3. Cascade parameters

The diffusion factor takes the pitch (`spacing`) and chord, so the solidity is
`sigma = chord / spacing`. The de Haller number is the ratio of exit to inlet
relative velocity. No limit values are drawn here: they are correlation
constants and will be added with a cited source.

In [ ]:
chord = 0.05  # m
sigma = np.linspace(0.8, 2.0, 25)
df_sigma = np.array(
    [diffusion_factor(w_1, w_2, c_theta_1, c_theta_2, chord / s, chord) for s in sigma]
)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(sigma, df_sigma)
ax.set_xlabel("solidity sigma = chord / pitch [-]")
ax.set_ylabel("diffusion factor [-]")
ax.set_title("Same triangle, varying solidity")
fig.tight_layout()
plt.show()

print(f"de Haller number for this triangle: {de_haller_number(w_1, w_2):.3f}")

In [ ]:
sigma_fixed = 1.2
c_theta_2_sweep = np.linspace(100.0, 300.0, 41)
w_2_sweep = np.hypot(c_x, u - c_theta_2_sweep)
loading = (c_theta_2_sweep - c_theta_1) / u

df_loading = np.array(
    [
        diffusion_factor(w_1, w2, c_theta_1, ct2, chord / sigma_fixed, chord)
        for w2, ct2 in zip(w_2_sweep, c_theta_2_sweep, strict=True)
    ]
)
dh_loading = np.array([de_haller_number(w_1, w2) for w2 in w_2_sweep])

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(loading, df_loading, label="diffusion factor")
ax.plot(loading, dh_loading, label="de Haller number")
ax.set_xlabel("loading coefficient psi [-]")
ax.set_ylabel("[-]")
ax.set_title("Fixed inlet triangle, solidity 1.2, increasing exit swirl")
ax.legend()
fig.tight_layout()
plt.show()

Raising the loading increases the diffusion factor and reduces the de Haller
number: the blade row is asked to decelerate the relative flow more.

## 4. What is not covered yet

- No validation against literature data (planned: `tests/validation/`).
- Equation numbers in the docstrings are still marked `TODO(source)`.
- Incidence, deviation and loss models are covered in later modules.